In [ ]:
# First run this cell
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from datetime import datetime
import math

The __self-attention__ computation is at the core of the Transformer architecture. It is important to get this computation efficient (i.e. vectorized), since it involves many matrix operations that would be very slow if implemented by Python loops.

The input to the self-attention computation is a tensor containing a vector for each input token, and the output is a tensor of the same dimensions, containing the contextualized versions of the input tokens (see the textbook, chapter 8.1).

Your task is to fill in the missing pieces below. Look for "REPLACE WITH YOUR CODE" and "YOUR CODE HERE".


In [ ]:
class SelfAttention(nn.Module):
    """
    Computes self-attention according to Vashwani et al, 2017.
    """

    def __init__(self, vector_dim, block_size, is_causal=False):
        """
        vector_dim = the dimension of the input and output vectors
        block_size = max number of tokens in the input 
        is_causal is True if tokens only can be influenced by preceding
                  tokens (typical in generative applications)
        """
        super().__init__()
        self.vector_dim = vector_dim
        self.is_causal = is_causal
        self.wq = nn.Linear(vector_dim, vector_dim, bias=False)
        self.wk = nn.Linear(vector_dim, vector_dim, bias=False)
        self.wv = nn.Linear(vector_dim, vector_dim, bias=False)
        self.wo = nn.Linear(vector_dim, vector_dim, bias=False)
        if self.is_causal:
            # The 'causal mask' is a lower-left triangular matrix of 1s, wrapped in
            # the outermost dimension(s) (which is the batch and, in the multihead
            # case, the number_of_heads dimension)
            causal_mask = torch.tril(torch.ones(block_size, block_size)).unsqueeze(0)
            # The next line creates a buffer 'self.mask'. Using a buffer rather than
            # a parameter ensures it will be moved to the GPU along with parameters,
            # but it won't be changed during training.
            self.register_buffer("mask", causal_mask)

    def compute_attention(self, q, k, v):
        # Shape of the tensors are (B,S,D) = (batch size, seq length, attention dim)
        # In the single-head case, attention_dim = vector_dim

        # YOUR CODE HERE
        
        if self.is_causal:
            pass  # REPLACE WITH YOUR CODE 

        # YOUR CODE HERE
    
        return values

    def forward(self, x):
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        values = self.compute_attention(q, k, v)
        out = self.wo(values)
        return out


In [ ]:
seed = 4224
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 1
block_size = 32
seq_len = 20 
vector_dim = 64
attention = SelfAttention(vector_dim, block_size, is_causal=False)
data = torch.rand(batch_size, seq_len, vector_dim)
result = attention(data)
print( "Without causal mask" )
print(f'{result[0][7][7].detach().item():.4f}' == '0.0342')
print(f'{result[0][8][8].detach().item():.4f}' == '0.2462')
print(f'{result[0][9][9].detach().item():.4f}' == '-0.1948')
print(f'{result[0][10][10].detach().item():.4f}' == '0.0360')
attention = SelfAttention(vector_dim, block_size, is_causal=True)
result = attention(data)
print( "With causal mask" )
print(f'{result[0][7][7].detach().item():.4f}' == '0.0154')
print(f'{result[0][8][8].detach().item():.4f}' == '-0.2425')
print(f'{result[0][9][9].detach().item():.4f}' == '0.3206')
print(f'{result[0][10][10].detach().item():.4f}' == '0.2764')

In [ ]:
class MultiHeadSelfAttention(SelfAttention):
    """
    Computes self-attention with multiple heads according to Vashwani et al, 2017.
    """

    def __init__(self, vector_dim, n_heads, block_size, is_causal=False):
        """
        vector_dim = the dimension of the input and output vectors
        n_heads = the number_of_attention_heads. It has to be a divisor of vector_dim
        block_size = max number of tokens in the input 
        is_causal is True if tokens only can be influenced by preceding
                  tokens (typical in generative applications)
        """   
        super().__init__(vector_dim, block_size, is_causal)
        self.att_dim = vector_dim//n_heads
        self.n_heads = n_heads

    def reshape_for_multihead_attention(self, x):
        """
        x has the shape (batch_size, seq_length, vector_dim)

        We want to split the representation of each token into 'number_of_heads'
        parts and treat each part separately. Thus, we need the returned tensor
        to have shape (batch_size, no_of_heads, seq_length, att_dim)
        """

        # YOUR CODE HERE

        return None # REPLACE WITH YOUR CODE

    def reshape_after_multihead_attention(self, x):
        """
        x has the shape (batch_size, no_of_heads, seq_length, att_dim)

        For each token, we now want to bring together the representation coming
        from each head. The returned token should have the shape:
        (batch_size, seq_length, vector_dim)
        """

        # YOUR CODE HERE

        return None # REPLACE WITH YOUR CODE

    def forward(self, x):
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        q = self.reshape_for_multihead_attention(q)
        k = self.reshape_for_multihead_attention(k)
        v = self.reshape_for_multihead_attention(v)
        values = self.compute_attention(q, k, v)
        values = self.reshape_after_multihead_attention(values)
        out = self.wo(values)
        return out

In [ ]:
"""
Test the MultiHeadSelfAttention class. Make sure that you first run the cell above!
"""
seed = 4224
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 1
block_size = 32
seq_len = 20
vector_dim = 64
no_of_heads = 2

attention = MultiHeadSelfAttention(vector_dim, no_of_heads, block_size, is_causal=False)
data = torch.rand(batch_size, seq_len, vector_dim)

print("Check that reshaping works:")
desired_shape = torch.Size([batch_size, attention.n_heads, seq_len, att_dim])
reshaped_data = attention.reshape_for_multihead_attention(data)
print( reshaped_data.shape == desired_shape )
print( (data[0][9][33] == reshaped_data[0][1][9][1]).detach().item() )
rereshaped_data = attention.reshape_after_multihead_attention(reshaped_data)
print( data.shape == rereshaped_data.shape )
print( torch.all(data == rereshaped_data).detach().item() )

result = attention(data)
print ("Multihead attention without causal mask")
print(f'{result[0][7][7].detach().item():.4f}' == '0.0336')
print(f'{result[0][8][8].detach().item():.4f}'== '0.2456')
print(f'{result[0][9][9].detach().item():.4f}' == '-0.1955')
print(f'{result[0][10][10].detach().item():.4f}' == '0.0356')

attention = MultiHeadSelfAttention(vector_dim, no_of_heads, block_size, is_causal=True)
result = attention(data)
print ("Multihead attention with causal mask")
print(f'{result[0][7][7].detach().item():.4f}' == '0.0147')
print(f'{result[0][8][8].detach().item():.4f}'== '-0.2433')
print(f'{result[0][9][9].detach().item():.4f}' == '0.3209')
print(f'{result[0][10][10].detach().item():.4f}' == '0.2763')